## RAG Application Using Type Sense

In [1]:
import typesense

We need to create the typesense collection. 

In the webiste https://cloud.typesense.org/clusters/e75hlonsy61pdrwjp after the cluster is done being initialized , we also need to go ahead and create a collection in the website itself.

The collection will be the place where we'll be storing the entire vector store.

In order to create the collection or in order to access this particular DB, what we need to do is that we need to go ahead and create a typesense client.

Why are we specifically creating this client so that we will be able to communicate with our typesense cloud platform.

How do we decide what host we really need to use?
Let's say you want to just go ahead and use local then we'll say 'host': "local", then you can provide any port which is available in your local device say 8080, and if we'are using local then we can just use hhtp, so this is basically the local configuration and here you also don't require any API key, say 'api_key':'kfl' this is more than just sufficient. 

How does a schema looks like?

Let's say if I have some kind of data now in that particular data I may have the fields like name I may have fields like type, authors, it can be different-different information, it can have information like publicationa_date, rating, average_rating, etc.

We take one example - in json format which is basically coming from the API, inside the schema looks like it has title, author, publication_date, rating, image_url,etc.

So let's say in my json, I have all the specific values and I really want to convert this into a RAG application and also store this information in our typesense cloud.

In [2]:
client=typesense.Client({
  'nodes': [{
    'host': 'e75hlonsy61pdrwjp-1.a1.typesense.net',  # For Typesense Cloud use xxx.a1.typesense.net
    'port': '443',       # For Typesense Cloud use 443
    'protocol': 'https'    # For Typesense Cloud use https
  }],
  'api_key':'MCN3Fc2Prj7Oh0EwS40YImF2kcrU9szr',
  'connection_timeout_seconds': 2
})

## Now we'll create our collection 
## to create a collection first of all we need to provide a schema
# this is one example 
books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
## The reason of setting up this particular schema is that we are going to go ahead and create our own collection by using the schema and then we will go ahead and insert some information out there. 

## Like afetr creating the schema whatever things are available over here we'll try to insert it inside our typesense cloud.
print(client.collections.create(books_schema))

ObjectAlreadyExists: [Errno 409] A collection with name `books` already exists.

We can go ahead in typesense website in our collection to see whether it's been updated there or not.

But still till now we won't be able to see any kind of data there.

But there you can very clearly see that in a very easy way , what did we do out of this particular code. In the particular code, you'll be able to see that I created my client , I created my book schema and then we used client.collection.create(bookz_schema) wherein we created the entire collection.

In [3]:
client

Now we will try to read this JSONal file (books.json) and then we will try to import all this particular information inside our collection.

In [4]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

WriteTimeout: The write operation timed out

Now check the website the data has been updated.

With the help of typesense cloud we are able to just go ahead and upload all the records and all the records is shown there in collection. All the indormation is easily uploaded you can go and easily search there , since typesense supports faster search you'll be able to probably do semantic search, you'll be able to do vector search and we'll go ahead and see the search. 

The best part about the search is that we can query by title, author, we can query by any kind of parameters, we can apply 'facet & filter by', and all.

Now in the coding side how we are gonna apply the search parameters.( We'll try to put criteria just like in the webside)

In [5]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

The most amazing part is see how fast this is.

We can do some filtering criteria also

In [6]:
search_parameters = {
  'q'         : 'harry potter',
  'query_by'  : 'title',
  'filter_by' : 'publication_year:<1998',
  'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [7]:
search_parameters = {
  'q'         : 'experyment',
  'query_by'  : 'title',
  'facet_by'  : 'authors',
  'sort_by'   : 'average_rating:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': ' Käthe Mazur',
     'value': ' Käthe Mazur'},
    {'count': 1, 'highlighted': 'Mahatma Gandhi', 'value': 'Mahatma Gandhi'},
    {'count': 1, 'highlighted': 'Gretchen Rubin', 'value': 'Gretchen Rubin'},
    {'count': 1,
     'highlighted': 'James Patterson',
     'value': 'James Patterson'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 4}}],
 'found': 3,
 'hits': [{'document': {'authors': ['James Patterson'],
    'average_rating': 4.08,
    'id': '569',
    'image_url': 'https://images.gr-assets.com/books/1339277875m/13152.jpg',
    'publication_year': 2005,
    'ratings_count': 172302,
    'title': 'The Angel Experiment'},
   'highlight': {'title': {'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}],
   'text_match': 5787300

This is the context.

At the end of the day, we're hitting the vector DB and getting the context and now we can just give this context to our LLM and probably generate any kind of output that we want.

What we did till here can we also so this with the help of langchain?

In [8]:
### Langchain + Typsense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense      # typesense is available in the form of vector store also --> u can do it local and can do it in cloud, its up to you 
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import os
os.environ["GROQ_API_KEY"] = "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"

There is one file google.txt. We'll try to store the entire information of this txt file with the help of langchain into our typesense cloud.

In [10]:
loader = TextLoader(
    "Google.txt",
    encoding="utf-8",
    autodetect_encoding=True
)                                   ## here we are doing data ingestion 
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=800, chunk_overlap=100)       # here we are doing preprocessing
docs = text_splitter.split_documents(documents)         # here we are splitting the documents 

embeddings = HuggingFaceEmbeddings()            # with the help of this HuggingFaceEmbeddings() embeddings we will go ahead and apply the embeddings

Created a chunk of size 949, which is longer than the specified 800
Created a chunk of size 922, which is longer than the specified 800
Created a chunk of size 892, which is longer than the specified 800
Created a chunk of size 825, which is longer than the specified 800
Created a chunk of size 921, which is longer than the specified 800
Created a chunk of size 830, which is longer than the specified 800
Created a chunk of size 1055, which is longer than the specified 800
Created a chunk of size 874, which is longer than the specified 800
C:\Users\Anshita\AppData\Local\Temp\ipykernel_28648\2477447340.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFace

Now we'll see how we can use typesense along with langchain.

In [11]:
# docsearch=Typesense.from_documents(
#     docs,
#     embeddings,
#     typesense_client_params={
#     'host': 'e75hlonsy61pdrwjp-1.a1.typesense.net',  # For Typesense Cloud use xxx.a1.typesense.net
#     'port': '443',       # For Typesense Cloud use 443
#     'protocol': 'https',    # For Typesense Cloud use https
#     'typesense_api_key':'MCN3Fc2Prj7Oh0EwS40YImF2kcrU9szr',
#     'typesense_collection_name':"lang-chain"
# },
    
# )

In [12]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,

    typesense_client_params={
        "host": "e75hlonsy61pdrwjp-1.a1.typesense.net",
        "port": "443",
        "protocol": "https",
        "connection_timeout_seconds": 120
    },

    typesense_api_key="MCN3Fc2Prj7Oh0EwS40YImF2kcrU9szr",

    typesense_collection_name="lang-chain"
)

In [13]:
query = "What is artificial intelligence"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

164. Edwards, Benj (May 15, 2024). "Google unveils Veo, a high-definition AI video generator
that may rival Sora" (https://arstechnica.com/information-technology/2024/05/google-unveils
-veo-a-high-definition-ai-video-generator-that-may-rival-sora/). Ars Technica. Retrieved
September 25, 2024.

165. Peters, Jay (May 21, 2025). "Google has a new tool to help detect AI-generated content" (htt
ps://www.theverge.com/news/672013/google-synthid-detector-ai-generated-content-waterm
ark-i-o-2025). The Verge. Retrieved May 23, 2025.

166. Orland, Kyle (September 23, 2024). "Fake AI "podcasters" are reviewing my book and it's
freaking me out" (https://arstechnica.com/ai/2024/09/fake-ai-podcasters-are-reviewing-my-b
ook-and-its-freaking-me-out/). Ars Technica. Retrieved September 25, 2024.


In [14]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001CF802BC1A0>, search_kwargs={})

In [15]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': 'Google.txt'}, page_content='92. Helgren, Chris (January 27, 2014). "Google to buy artificial intelligence company DeepMind"\n(https://www.reuters.com/article/us-google-deepmind-idUSBREA0Q03220140127). Reuters.\nArchived (https://web.archive.org/web/20140127042513/http://www.reuters.com/article/201\n4/01/27/us-google-deepmind-idUSBREA0Q03220140127) from the original on January 27,\n2014. Retrieved January 27, 2014.')

We can also do this in local with

docsearch = Typesense.from_documents(
    docs,
    embeddings,

    typesense_client_params={
        "host": "local",
        "port": "443",
        "protocol": "http",
        "connection_timeout_seconds": 120
    },
    typesense_collection_name="lang-chain"
)